In [23]:
import os
import glob
import gc
import warnings
warnings.filterwarnings("ignore")
from typing import List, Tuple
import logging
logging.basicConfig(level=logging.CRITICAL)


import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, ParameterGrid
import lightgbm as lgb



In [24]:

# DATA_DIR


def find_data_dir() -> str:
    for d in glob.glob("/kaggle/input/*"):
        if (
            os.path.exists(os.path.join(d, "sample_submission.csv"))
            and os.path.exists(os.path.join(d, "train_log.csv"))
            and os.path.exists(os.path.join(d, "test_log.csv"))
        ):
            return d
    return "/kaggle/input/datadsetforml/dataset"


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

SPLIT_DIRS = sorted(glob.glob(os.path.join(DATA_DIR, "split_*")))
print("Found splits:", len(SPLIT_DIRS))

RANDOM_STATE = 42
N_FOLDS = 5


DATA_DIR = /kaggle/input/datadsetforml/dataset
Found splits: 20


In [25]:
# Metadata
train_log = pd.read_csv(os.path.join(DATA_DIR, "train_log.csv"))
test_log = pd.read_csv(os.path.join(DATA_DIR, "test_log.csv"))


In [26]:

# De-extinction (EBV)
EFF_WL = {"u": 3641.0, "g": 4704.0, "r": 6155.0, "i": 7504.0, "z": 8695.0, "y": 10056.0}  # Angstrom
FILTERS = ["u", "g", "r", "i", "z", "y"]
RV = 3.1


def ccm89_A_lambda(wave_angstrom: np.ndarray, A_V: np.ndarray, R_V: float = 3.1) -> np.ndarray:
    wave_micron = wave_angstrom * 1e-4
    x = 1.0 / wave_micron  # micron^-1
    a = np.zeros_like(x, dtype=np.float64)
    b = np.zeros_like(x, dtype=np.float64)

    m_nir = x < 1.1
    a[m_nir] = 0.574 * (x[m_nir] ** 1.61)
    b[m_nir] = -0.527 * (x[m_nir] ** 1.61)

    m_opt = (x >= 1.1) & (x <= 3.3)
    y = x[m_opt] - 1.82
    a[m_opt] = (
        1
        + 0.17699 * y
        - 0.50447 * y**2
        - 0.02427 * y**3
        + 0.72085 * y**4
        + 0.01979 * y**5
        - 0.77530 * y**6
        + 0.32999 * y**7
    )
    b[m_opt] = (
        1.41338 * y
        + 2.28305 * y**2
        + 1.07233 * y**3
        - 5.38434 * y**4
        - 0.62251 * y**5
        + 5.30260 * y**6
        - 2.09002 * y**7
    )

    A_over_AV = a + b / R_V
    return A_V * A_over_AV


def build_corr_table(log_all: pd.DataFrame) -> pd.DataFrame:
    obj = log_all[["object_id", "EBV"]].drop_duplicates()
    A_V = obj["EBV"].astype(np.float64).values * RV
    rows = []
    for f in FILTERS:
        w = np.float64(EFF_WL[f])
        wl = np.full(len(obj), w, dtype=np.float64)
        A_lam = ccm89_A_lambda(wl, A_V, R_V=RV)
        corr = (10.0 ** (A_lam / 2.5)).astype(np.float32)
        rows.append(pd.DataFrame({"object_id": obj["object_id"].values, "filter": f, "corr": corr}))
    return pd.concat(rows, ignore_index=True)


log_all = pd.concat([train_log[["object_id", "EBV"]], test_log[["object_id", "EBV"]]], ignore_index=True)
corr_long = build_corr_table(log_all)
del log_all
gc.collect()


25394

In [27]:

# Feature engineering
LC_DTYPES = {
    "object_id": "string",
    "Time (MJD)": "float32",
    "Flux": "float32",
    "Flux_err": "float32",
    "Filter": "category",
}


def extract_features_from_lc(lc: pd.DataFrame) -> pd.DataFrame:
    df = lc.rename(
        columns={"Time (MJD)": "mjd", "Flux": "flux", "Flux_err": "flux_err", "Filter": "filter"}
    ).copy()
    df["filter"] = df["filter"].astype("string")

    # extinction correction
    df = df.merge(corr_long, on=["object_id", "filter"], how="left")
    df["corr"] = df["corr"].fillna(1.0).astype(np.float32)
    df["flux"] = (df["flux"] * df["corr"]).astype(np.float32)
    df["flux_err"] = (df["flux_err"] * df["corr"]).astype(np.float32)

    df["flux_asinh"] = np.arcsinh(df["flux"].astype(np.float64)).astype(np.float32)
    df["snr"] = (np.abs(df["flux"]) / np.clip(df["flux_err"], 1e-6, None)).astype(np.float32)
    df["abs_flux"] = np.abs(df["flux"]).astype(np.float32)
    df["flux_sq"] = (df["flux"].astype(np.float64) ** 2).astype(np.float32)
    df["snr_sq"] = (df["snr"].astype(np.float64) ** 2).astype(np.float32)
    df["flux_err_abs"] = df["flux_err"].abs().astype(np.float32)
    df["rel_err"] = (df["flux_err_abs"] / df["abs_flux"].clip(1e-6, None)).astype(np.float32)
    df["w"] = (1.0 / np.square(df["flux_err_abs"].clip(1e-3, None))).astype(np.float32)

    df = df.sort_values(["object_id", "filter", "mjd"])
    g = df.groupby(["object_id", "filter"], sort=False)

    df["dt"] = g["mjd"].diff().astype(np.float32)
    df["flux_prev"] = g["flux"].shift().astype(np.float32)
    df["trap"] = ((df["flux"] + df["flux_prev"]) * 0.5 * df["dt"]).astype(np.float32)
    df["trap_abs"] = np.abs(df["trap"]).astype(np.float32)
    df["trap_pos"] = df["trap"].clip(lower=0).astype(np.float32)
    df["trap_neg"] = (-df["trap"]).clip(lower=0).astype(np.float32)
    df["sign"] = np.sign(df["flux"].fillna(0)).astype(np.int8)
    df["sign_prev"] = g["sign"].shift()
    df["sign_change"] = np.where(df["sign_prev"].isna(), 0, (df["sign"] != df["sign_prev"]).astype(np.int8))

    base = g.agg(
        n_obs=("mjd", "count"),
        t_min=("mjd", "min"),
        t_max=("mjd", "max"),
        flux_mean=("flux", "mean"),
        flux_std=("flux", "std"),
        flux_min=("flux", "min"),
        flux_max=("flux", "max"),
        flux_median=("flux", "median"),
        abs_flux_mean=("abs_flux", "mean"),
        abs_flux_std=("abs_flux", "std"),
        asinh_mean=("flux_asinh", "mean"),
        asinh_std=("flux_asinh", "std"),
        err_mean=("flux_err", "mean"),
        err_std=("flux_err", "std"),
        err_median=("flux_err", "median"),
        err_min=("flux_err", "min"),
        err_max=("flux_err", "max"),
        rel_err_mean=("rel_err", "mean"),
        rel_err_median=("rel_err", "median"),
        rel_err_max=("rel_err", "max"),
        snr_max=("snr", "max"),
        snr_mean=("snr", "mean"),
        snr_std=("snr", "std"),
        snr_min=("snr", "min"),
    ).reset_index()

    q01 = g["flux"].quantile(0.01).rename("flux_q01")
    q10 = g["flux"].quantile(0.10).rename("flux_q10")
    q25 = g["flux"].quantile(0.25).rename("flux_q25")
    q75 = g["flux"].quantile(0.75).rename("flux_q75")
    q90 = g["flux"].quantile(0.90).rename("flux_q90")
    q99 = g["flux"].quantile(0.99).rename("flux_q99")
    rel_err_p90 = g["rel_err"].quantile(0.90).rename("rel_err_p90")

    max_gap = df.groupby(["object_id", "filter"], sort=False)["dt"].max().rename("max_gap")
    med_gap = df.groupby(["object_id", "filter"], sort=False)["dt"].median().rename("med_gap")
    dt_std = df.groupby(["object_id", "filter"], sort=False)["dt"].std().rename("dt_std")
    dt_mean = df.groupby(["object_id", "filter"], sort=False)["dt"].mean().rename("dt_mean")
    dt_p10 = g["dt"].quantile(0.10).rename("dt_p10")
    dt_p90 = g["dt"].quantile(0.90).rename("dt_p90")
    area = df.groupby(["object_id", "filter"], sort=False)["trap"].sum().rename("area")
    area_abs = df.groupby(["object_id", "filter"], sort=False)["trap_abs"].sum().rename("area_abs")
    area_pos = df.groupby(["object_id", "filter"], sort=False)["trap_pos"].sum().rename("area_pos")
    area_neg = df.groupby(["object_id", "filter"], sort=False)["trap_neg"].sum().rename("area_neg")
    abs_flux_sum = df.groupby(["object_id", "filter"], sort=False)["abs_flux"].sum().rename("abs_flux_sum")
    flux_wt_mjd = (df["mjd"] * df["abs_flux"]).groupby([df["object_id"], df["filter"]]).sum().rename("flux_wt_mjd")
    flux_com = (flux_wt_mjd / abs_flux_sum.replace(0, np.nan)).rename("flux_com")
    snr_wt_mjd = (df["mjd"] * df["snr"]).groupby([df["object_id"], df["filter"]]).sum().rename("snr_wt_mjd")
    snr_sum_pos = df.groupby([df["object_id"], df["filter"]])["snr"].sum().rename("snr_sum_pos")
    snr_com = (snr_wt_mjd / snr_sum_pos.replace(0, np.nan)).rename("snr_com")
    pos_frac = (df["flux"] > 0).groupby([df["object_id"], df["filter"]]).mean().rename("pos_frac")
    snr_gt3 = (df["snr"] > 3).groupby([df["object_id"], df["filter"]]).sum().rename("snr_gt3")
    snr_gt5 = (df["snr"] > 5).groupby([df["object_id"], df["filter"]]).sum().rename("snr_gt5")
    snr_gt8 = (df["snr"] > 8).groupby([df["object_id"], df["filter"]]).sum().rename("snr_gt8")
    snr_p10 = g["snr"].quantile(0.10).rename("snr_p10")
    snr_p25 = g["snr"].quantile(0.25).rename("snr_p25")
    snr_p75 = g["snr"].quantile(0.75).rename("snr_p75")
    snr_p90 = g["snr"].quantile(0.90).rename("snr_p90")
    snr_p99 = g["snr"].quantile(0.99).rename("snr_p99")
    snr_median = g["snr"].median().rename("snr_median")
    snr_sum = g["snr"].sum().rename("snr_sum")
    snr_energy = g["snr_sq"].sum().rename("snr_energy")
    flux_energy = g["flux_sq"].sum().rename("flux_energy")
    sign_changes = df.groupby(["object_id", "filter"], sort=False)["sign_change"].sum().rename("sign_changes")
    flux_skew = g["flux"].skew().rename("flux_skew")
    flux_kurt = g["flux"].agg(pd.Series.kurt).rename("flux_kurt")
    snr_kurt = g["snr"].agg(pd.Series.kurt).rename("snr_kurt")
    w_sum = df.groupby(["object_id", "filter"], sort=False)["w"].sum().rename("w_sum")
    w_flux_sum = (df["flux"] * df["w"]).groupby([df["object_id"], df["filter"]]).sum().rename("w_flux_sum")
    w_flux2_sum = ((df["flux"].astype(np.float64) ** 2) * df["w"]).groupby([df["object_id"], df["filter"]]).sum().rename(
        "w_flux2_sum"
    )
    flux_wmean = (w_flux_sum / w_sum.replace(0, np.nan)).rename("flux_wmean")
    flux_wvar = (w_flux2_sum / w_sum.replace(0, np.nan) - (flux_wmean**2)).rename("flux_wvar")
    flux_wstd = np.sqrt(flux_wvar.clip(lower=0)).rename("flux_wstd")

    extra = pd.concat(
        [
            q01,
            q10,
            q25,
            q75,
            q90,
            q99,
            max_gap,
            med_gap,
            dt_std,
            dt_mean,
            dt_p10,
            dt_p90,
            area,
            area_abs,
            area_pos,
            area_neg,
            abs_flux_sum,
            flux_com,
            snr_com,
            pos_frac,
            snr_gt3,
            snr_gt5,
            snr_gt8,
            snr_p10,
            snr_p25,
            snr_p75,
            snr_p90,
            snr_p99,
            snr_median,
            snr_sum,
            snr_energy,
            snr_kurt,
            flux_energy,
            sign_changes,
            flux_skew,
            flux_kurt,
            rel_err_p90,
            w_sum,
            flux_wmean,
            flux_wstd,
        ],
        axis=1,
    ).reset_index()

    feat = base.merge(extra, on=["object_id", "filter"], how="left")
    feat["t_span"] = feat["t_max"] - feat["t_min"]
    feat["amp"] = feat["flux_max"] - feat["flux_min"]
    feat["amp_to_std"] = feat["amp"] / (feat["flux_std"].fillna(0) + 1e-3)
    feat["flux_iqr"] = (feat["flux_q75"] - feat["flux_q25"]).astype(np.float32)
    feat["flux_range_p90_p10"] = (feat["flux_q90"] - feat["flux_q10"]).astype(np.float32)
    feat["flux_range_p99_p01"] = (feat["flux_q99"] - feat["flux_q01"]).astype(np.float32)
    feat["area_ratio"] = (
        feat["area"] / feat["area_abs"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["area_over_span"] = (
        feat["area"] / feat["t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["area_pos"] = feat["area_pos"].fillna(0).astype(np.float32)
    feat["area_neg"] = feat["area_neg"].fillna(0).astype(np.float32)
    feat["area_pos_frac"] = (
        feat["area_pos"] / feat["area_abs"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["area_neg_frac"] = (
        feat["area_neg"] / feat["area_abs"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["area_pos_neg_ratio"] = (
        feat["area_pos"] / (feat["area_neg"] + 1e-3)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["abs_flux_sum"] = feat["abs_flux_sum"].fillna(0).astype(np.float32)
    feat["flux_com"] = feat["flux_com"].fillna(0).astype(np.float32)
    feat["snr_com"] = feat["snr_com"].fillna(0).astype(np.float32)
    feat["flux_com_rel"] = (
        (feat["flux_com"] - feat["t_min"]) / feat["t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["snr_com_rel"] = (
        (feat["snr_com"] - feat["t_min"]) / feat["t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["flux_skew"] = feat["flux_skew"].fillna(0).astype(np.float32)
    feat["dt_std"] = feat["dt_std"].fillna(0).astype(np.float32)
    feat["cadence_cv"] = (
        feat["dt_std"] / feat["med_gap"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["flux_cv"] = (
        feat["flux_std"] / (feat["flux_mean"].abs() + 1e-3)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["flux_rsd"] = (
        feat["flux_std"] / (feat["flux_median"].abs() + 1e-3)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["abs_flux_cv"] = (
        feat["abs_flux_std"] / (feat["abs_flux_mean"].abs() + 1e-3)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["sign_changes"] = feat["sign_changes"].fillna(0).astype(np.float32)
    feat["sign_change_rate"] = (
        feat["sign_changes"] / (feat["n_obs"] - 1).clip(lower=1)
    ).astype(np.float32)
    feat["snr_gt8"] = feat["snr_gt8"].fillna(0).astype(np.float32)
    feat["snr_sum"] = feat["snr_sum"].fillna(0).astype(np.float32)
    feat["snr_energy"] = feat["snr_energy"].fillna(0).astype(np.float32)
    feat["snr_energy_norm"] = (
        feat["snr_energy"] / feat["n_obs"].clip(lower=1)
    ).astype(np.float32)
    feat["flux_energy"] = feat["flux_energy"].fillna(0).astype(np.float32)
    feat["flux_energy_norm"] = (
        feat["flux_energy"] / feat["n_obs"].clip(lower=1)
    ).astype(np.float32)
    feat["snr_std"] = feat["snr_std"].fillna(0).astype(np.float32)
    feat["err_std"] = feat["err_std"].fillna(0).astype(np.float32)
    feat["snr_cv"] = (
        feat["snr_std"] / feat["snr_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["err_cv"] = (
        feat["err_std"] / feat["err_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["rel_err_p90"] = feat["rel_err_p90"].fillna(0).astype(np.float32)
    feat["flux_wmean"] = feat["flux_wmean"].fillna(0).astype(np.float32)
    feat["flux_wstd"] = feat["flux_wstd"].fillna(0).astype(np.float32)
    feat["flux_wcv"] = (
        feat["flux_wstd"] / (feat["flux_wmean"].abs() + 1e-3)
    ).astype(np.float32)
    feat["snr_range_p90_p10"] = (
        (feat["snr_p90"] - feat["snr_p10"]).astype(np.float32)
    ).fillna(0)
    feat["dt_p90_p10"] = (
        (feat["dt_p90"] - feat["dt_p10"]).astype(np.float32)
    ).fillna(0)

    det = df[(df["snr"] >= 3) & (df["flux"] > 0)].copy()
    if len(det) > 0:
        gdet = det.groupby(["object_id", "filter"], sort=False)
        det_feat = gdet.agg(
            det_n=("mjd", "count"),
            det_t_min=("mjd", "min"),
            det_t_max=("mjd", "max"),
            det_flux_max=("flux", "max"),
            det_flux_mean=("flux", "mean"),
            det_snr_max=("snr", "max"),
            det_snr_mean=("snr", "mean"),
        )
        det_feat["det_span"] = (det_feat["det_t_max"] - det_feat["det_t_min"]).astype(np.float32)
        det_feat = det_feat.drop(columns=["det_t_min", "det_t_max"]).reset_index()
    else:
        det_feat = pd.DataFrame(
            columns=[
                "object_id",
                "filter",
                "det_n",
                "det_flux_max",
                "det_flux_mean",
                "det_snr_max",
                "det_snr_mean",
                "det_span",
            ]
        )

    feat = feat.merge(det_feat, on=["object_id", "filter"], how="left")
    feat["det_n"] = feat["det_n"].fillna(0).astype(np.float32)
    feat["det_span"] = feat["det_span"].fillna(0).astype(np.float32)
    feat["det_frac"] = (feat["det_n"] / feat["n_obs"].clip(lower=1)).astype(np.float32)
    feat["det_span_ratio"] = (
        feat["det_span"] / feat["t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["det_flux_max_to_amp"] = (
        feat["det_flux_max"] / feat["amp"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

    df_nonan = df.dropna(subset=["flux"])
    if len(df_nonan) > 0:
        g2 = df_nonan.groupby(["object_id", "filter"], sort=False)
        peak_idx = g2["flux"].idxmax().dropna()
        peak = df_nonan.loc[peak_idx, ["object_id", "filter", "mjd", "flux"]].rename(
            columns={"mjd": "peak_mjd", "flux": "peak_flux"}
        )
        first = g2[["mjd", "flux"]].first().reset_index().rename(columns={"mjd": "first_mjd", "flux": "first_flux"})
        last = g2[["mjd", "flux"]].last().reset_index().rename(columns={"mjd": "last_mjd", "flux": "last_flux"})
        peak = peak.merge(first, on=["object_id", "filter"], how="left").merge(last, on=["object_id", "filter"], how="left")

        rise_den = (peak["peak_mjd"] - peak["first_mjd"]).replace(0, np.nan)
        decay_den = (peak["last_mjd"] - peak["peak_mjd"]).replace(0, np.nan)
        peak["rise_slope"] = (peak["peak_flux"] - peak["first_flux"]) / rise_den
        peak["decay_slope"] = (peak["last_flux"] - peak["peak_flux"]) / decay_den
        peak["peak_rel"] = peak["peak_mjd"] - feat.set_index(["object_id", "filter"]).loc[
            peak.set_index(["object_id", "filter"]).index, "t_min"
        ].values
        peak = peak[["object_id", "filter", "peak_mjd", "peak_flux", "first_flux", "last_flux", "rise_slope", "decay_slope", "peak_rel"]]
    else:
        peak = pd.DataFrame(
            columns=["object_id", "filter", "peak_mjd", "peak_flux", "first_flux", "last_flux", "rise_slope", "decay_slope", "peak_rel"]
        )

    feat = feat.merge(peak, on=["object_id", "filter"], how="left")
    feat["peak_phase"] = (
        feat["peak_rel"] / feat["t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["peak_flux_norm_amp"] = (
        feat["peak_flux"] / feat["amp"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["flux_trend_slope"] = (
        (feat["last_flux"] - feat["first_flux"]).astype(np.float32)
        / feat["t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    feat["rise_decay_ratio"] = (
        feat["rise_slope"] / feat["decay_slope"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

    if len(peak) > 0:
        dfp = df.merge(peak[["object_id", "filter", "peak_flux"]], on=["object_id", "filter"], how="left")
        for frac in [0.50, 0.75]:
            m = (dfp["peak_flux"] > 0) & (dfp["flux"] >= (frac * dfp["peak_flux"]))
            tmp = dfp[m].groupby(["object_id", "filter"], sort=False)["mjd"].agg(["min", "max"])
            tmp[f"width_{int(frac * 100)}"] = (tmp["max"] - tmp["min"]).astype(np.float32)
            tmp = tmp[[f"width_{int(frac * 100)}"]].reset_index()
            feat = feat.merge(tmp, on=["object_id", "filter"], how="left")
    else:
        feat["width_50"] = 0.0
        feat["width_75"] = 0.0

    for c in ["width_50", "width_75"]:
        if c in feat.columns:
            feat[c] = feat[c].fillna(0).astype(np.float32)
            feat[c + "_ratio"] = (
                feat[c] / feat["t_span"].replace(0, np.nan)
            ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

    wide = feat.pivot(index="object_id", columns="filter")
    wide.columns = [f"{filt}_{col}" for col, filt in wide.columns]
    wide = wide.reset_index()

    # ensure first/last/peak columns exist for color deltas
    for f in FILTERS:
        for suf in ["peak_flux", "first_flux", "last_flux", "peak_rel", "t_min"]:
            col = f"{f}_{suf}"
            if col not in wide.columns:
                wide[col] = np.nan
        wide[f"{f}_has"] = (wide.get(f"{f}_n_obs", 0).fillna(0) > 0).astype(np.int8)

    color_pairs = [("g", "r"), ("r", "i"), ("i", "z"), ("u", "g"), ("z", "y"), ("g", "i"), ("r", "z")]
    for a, b in color_pairs:
        wide[f"color_{a}{b}_asinh"] = np.arcsinh(wide[f"{a}_peak_flux"].astype(np.float64)) - np.arcsinh(
            wide[f"{b}_peak_flux"].astype(np.float64)
        )
        wide[f"color_{a}{b}_first_asinh"] = np.arcsinh(wide[f"{a}_first_flux"].astype(np.float64)) - np.arcsinh(
            wide[f"{b}_first_flux"].astype(np.float64)
        )
        wide[f"color_{a}{b}_last_asinh"] = np.arcsinh(wide[f"{a}_last_flux"].astype(np.float64)) - np.arcsinh(
            wide[f"{b}_last_flux"].astype(np.float64)
        )
        wide[f"color_{a}{b}_delta_peak_first"] = (wide[f"color_{a}{b}_asinh"] - wide[f"color_{a}{b}_first_asinh"]).astype(
            np.float32
        )
        wide[f"color_{a}{b}_delta_last_first"] = (
            wide[f"color_{a}{b}_last_asinh"] - wide[f"color_{a}{b}_first_asinh"]
        ).astype(np.float32)
        wide[f"color_{a}{b}_delta_peak_last"] = (
            wide[f"color_{a}{b}_asinh"] - wide[f"color_{a}{b}_last_asinh"]
        ).astype(np.float32)
        wide[f"dt_peak_{a}{b}"] = wide[f"{a}_peak_rel"] - wide[f"{b}_peak_rel"]

    peak_cols = [c for c in wide.columns if c.endswith("_peak_flux")]
    if peak_cols:
        peak_df = wide[peak_cols]
        wide["peak_flux_total"] = peak_df.sum(axis=1)
        wide["peak_flux_std"] = peak_df.std(axis=1)
        wide["peak_flux_range"] = peak_df.max(axis=1) - peak_df.min(axis=1)
        denom = wide["peak_flux_total"].replace(0, np.nan)
        for c in peak_cols:
            wide[c.replace("peak_flux", "peak_frac")] = (
                wide[c] / denom
            ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
        wide["peak_flux_ratio_max_min"] = (
            peak_df.max(axis=1) / peak_df.clip(lower=1e-3).min(axis=1)
        ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

    tmin_cols = [f"{f}_t_min" for f in FILTERS if f"{f}_t_min" in wide.columns]
    if tmin_cols:
        idx = wide[tmin_cols].idxmin(axis=1, skipna=True)
        wide["first_filter"] = idx.str.split("_", n=1).str[0]
    peak_cols_valid = [f"{f}_peak_flux" for f in FILTERS if f"{f}_peak_flux" in wide.columns]
    if peak_cols_valid:
        idx = wide[peak_cols_valid].idxmax(axis=1, skipna=True)
        wide["peak_filter"] = idx.str.split("_", n=1).str[0]

    df_all = df.sort_values(["object_id", "mjd"])
    gg = df_all.groupby("object_id", sort=False)
    df_all["dt_all"] = gg["mjd"].diff().astype(np.float32)
    df_all["flux_prev_all"] = gg["flux"].shift().astype(np.float32)
    df_all["trap_all"] = ((df_all["flux"] + df_all["flux_prev_all"]) * 0.5 * df_all["dt_all"]).astype(np.float32)
    df_all["trap_all_abs"] = np.abs(df_all["trap_all"]).astype(np.float32)
    df_all["trap_all_pos"] = df_all["trap_all"].clip(lower=0).astype(np.float32)
    df_all["trap_all_neg"] = (-df_all["trap_all"]).clip(lower=0).astype(np.float32)
    df_all["abs_flux_all"] = np.abs(df_all["flux"]).astype(np.float32)
    df_all["flux_err_abs"] = df_all["flux_err"].abs().astype(np.float32)
    df_all["rel_err"] = (df_all["flux_err_abs"] / df_all["abs_flux_all"].clip(1e-6, None)).astype(np.float32)
    df_all["w"] = (1.0 / np.square(df_all["flux_err_abs"].clip(1e-3, None))).astype(np.float32)
    df_all["sign_all"] = np.sign(df_all["flux"].fillna(0)).astype(np.int8)
    df_all["sign_prev_all"] = gg["sign_all"].shift()
    df_all["sign_change_all"] = np.where(
        df_all["sign_prev_all"].isna(), 0, (df_all["sign_all"] != df_all["sign_prev_all"]).astype(np.int8)
    )

    global_feat = gg.agg(
        all_n_obs=("mjd", "count"),
        all_t_min=("mjd", "min"),
        all_t_max=("mjd", "max"),
        all_flux_mean=("flux", "mean"),
        all_flux_std=("flux", "std"),
        all_flux_min=("flux", "min"),
        all_flux_max=("flux", "max"),
        all_flux_median=("flux", "median"),
        all_abs_flux_mean=("abs_flux_all", "mean"),
        all_abs_flux_std=("abs_flux_all", "std"),
        all_err_mean=("flux_err", "mean"),
        all_err_std=("flux_err", "std"),
        all_err_median=("flux_err", "median"),
        all_err_min=("flux_err", "min"),
        all_err_max=("flux_err", "max"),
        all_rel_err_mean=("rel_err", "mean"),
        all_rel_err_median=("rel_err", "median"),
        all_snr_max=("snr", "max"),
        all_snr_mean=("snr", "mean"),
        all_snr_std=("snr", "std"),
        all_snr_min=("snr", "min"),
        all_filters_n=("filter", "nunique"),
    ).reset_index()

    all_max_gap = df_all.groupby("object_id", sort=False)["dt_all"].max().rename("all_max_gap").reset_index()
    all_med_gap = df_all.groupby("object_id", sort=False)["dt_all"].median().rename("all_med_gap").reset_index()
    all_dt_std = df_all.groupby("object_id", sort=False)["dt_all"].std().rename("all_dt_std").reset_index()
    all_area = df_all.groupby("object_id", sort=False)["trap_all"].sum().rename("all_area").reset_index()
    all_area_abs = df_all.groupby("object_id", sort=False)["trap_all_abs"].sum().rename("all_area_abs").reset_index()
    all_area_pos = df_all.groupby("object_id", sort=False)["trap_all_pos"].sum().rename("all_area_pos").reset_index()
    all_area_neg = df_all.groupby("object_id", sort=False)["trap_all_neg"].sum().rename("all_area_neg").reset_index()
    all_pos_frac = (df_all["flux"] > 0).groupby(df_all["object_id"]).mean().rename("all_pos_frac").reset_index()
    all_sign_changes = df_all.groupby("object_id", sort=False)["sign_change_all"].sum().rename("all_sign_changes").reset_index()
    all_snr_sum = df_all.groupby("object_id", sort=False)["snr"].sum().rename("all_snr_sum").reset_index()
    all_snr_energy = df_all.groupby("object_id", sort=False)["snr_sq"].sum().rename("all_snr_energy").reset_index()
    all_flux_energy = df_all.groupby("object_id", sort=False)["flux_sq"].sum().rename("all_flux_energy").reset_index()
    all_abs_flux_sum = df_all.groupby("object_id", sort=False)["abs_flux_all"].sum().rename("all_abs_flux_sum").reset_index()
    all_w_sum = df_all.groupby("object_id", sort=False)["w"].sum().rename("all_w_sum").reset_index()
    all_w_flux_sum = (
        (df_all["flux"] * df_all["w"]).groupby(df_all["object_id"]).sum().rename("all_w_flux_sum").reset_index()
    )
    all_w_flux2_sum = (
        ((df_all["flux"].astype(np.float64) ** 2) * df_all["w"])
        .groupby(df_all["object_id"])
        .sum()
        .rename("all_w_flux2_sum")
        .reset_index()
    )
    all_w = all_w_sum.merge(all_w_flux_sum, on="object_id", how="left").merge(all_w_flux2_sum, on="object_id", how="left")
    all_w["all_flux_wmean"] = (all_w["all_w_flux_sum"] / all_w["all_w_sum"].replace(0, np.nan)).replace(
        [np.inf, -np.inf], np.nan
    )
    all_w["all_flux_wvar"] = (
        all_w["all_w_flux2_sum"] / all_w["all_w_sum"].replace(0, np.nan) - (all_w["all_flux_wmean"] ** 2)
    ).replace([np.inf, -np.inf], np.nan)
    all_w["all_flux_wstd"] = np.sqrt(all_w["all_flux_wvar"].clip(lower=0))
    all_w = all_w[["object_id", "all_w_sum", "all_flux_wmean", "all_flux_wstd"]]
    all_flux_wt_mjd = (
        (df_all["mjd"] * df_all["abs_flux_all"]).groupby(df_all["object_id"]).sum().rename("all_flux_wt_mjd").reset_index()
    )
    all_flux_com = all_flux_wt_mjd.merge(all_abs_flux_sum, on="object_id", how="left")
    all_flux_com["all_flux_com"] = (
        all_flux_com["all_flux_wt_mjd"] / all_flux_com["all_abs_flux_sum"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan)
    all_flux_com = all_flux_com[["object_id", "all_flux_com"]]
    all_snr_wt_mjd = (
        (df_all["mjd"] * df_all["snr"]).groupby(df_all["object_id"]).sum().rename("all_snr_wt_mjd").reset_index()
    )
    all_snr_com = all_snr_wt_mjd.merge(all_snr_sum, on="object_id", how="left")
    all_snr_com["all_snr_com"] = (
        all_snr_com["all_snr_wt_mjd"] / all_snr_com["all_snr_sum"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan)
    all_snr_com = all_snr_com[["object_id", "all_snr_com"]]
    all_snr_gt8 = (df_all["snr"] > 8).groupby(df_all["object_id"]).sum().rename("all_snr_gt8").reset_index()

    global_feat = (
        global_feat.merge(all_max_gap, on="object_id", how="left")
        .merge(all_med_gap, on="object_id", how="left")
        .merge(all_dt_std, on="object_id", how="left")
        .merge(all_area, on="object_id", how="left")
        .merge(all_area_abs, on="object_id", how="left")
        .merge(all_area_pos, on="object_id", how="left")
        .merge(all_area_neg, on="object_id", how="left")
        .merge(all_pos_frac, on="object_id", how="left")
        .merge(all_sign_changes, on="object_id", how="left")
        .merge(all_snr_sum, on="object_id", how="left")
        .merge(all_snr_energy, on="object_id", how="left")
        .merge(all_flux_energy, on="object_id", how="left")
        .merge(all_abs_flux_sum, on="object_id", how="left")
        .merge(all_w, on="object_id", how="left")
        .merge(all_flux_com, on="object_id", how="left")
        .merge(all_snr_com, on="object_id", how="left")
        .merge(all_snr_gt8, on="object_id", how="left")
    )
    global_feat["all_t_span"] = global_feat["all_t_max"] - global_feat["all_t_min"]
    global_feat["all_amp"] = global_feat["all_flux_max"] - global_feat["all_flux_min"]
    global_feat["all_area_ratio"] = (
        global_feat["all_area"] / global_feat["all_amp"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_area_over_span"] = (
        global_feat["all_area"] / global_feat["all_t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_area_balance"] = (
        global_feat["all_area"] / global_feat["all_area_abs"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_area_pos"] = global_feat["all_area_pos"].fillna(0).astype(np.float32)
    global_feat["all_area_neg"] = global_feat["all_area_neg"].fillna(0).astype(np.float32)
    global_feat["all_area_pos_frac"] = (
        global_feat["all_area_pos"] / global_feat["all_area_abs"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_area_neg_frac"] = (
        global_feat["all_area_neg"] / global_feat["all_area_abs"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_area_pos_neg_ratio"] = (
        global_feat["all_area_pos"] / (global_feat["all_area_neg"] + 1e-3)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_abs_flux_sum"] = global_feat["all_abs_flux_sum"].fillna(0).astype(np.float32)
    global_feat["all_flux_com"] = global_feat["all_flux_com"].fillna(0).astype(np.float32)
    global_feat["all_flux_com_rel"] = (
        (global_feat["all_flux_com"] - global_feat["all_t_min"]) / global_feat["all_t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_snr_com"] = global_feat["all_snr_com"].fillna(0).astype(np.float32)
    global_feat["all_snr_com_rel"] = (
        (global_feat["all_snr_com"] - global_feat["all_t_min"]) / global_feat["all_t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["filter_coverage"] = (global_feat["all_filters_n"] / len(FILTERS)).astype(np.float32)
    global_feat["all_cadence_cv"] = (
        global_feat["all_dt_std"].fillna(0) / global_feat["all_med_gap"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_flux_cv"] = (
        global_feat["all_flux_std"] / (global_feat["all_flux_mean"].abs() + 1e-3)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_abs_flux_cv"] = (
        global_feat["all_abs_flux_std"] / (global_feat["all_abs_flux_mean"].abs() + 1e-3)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_sign_changes"] = global_feat["all_sign_changes"].fillna(0).astype(np.float32)
    global_feat["all_sign_change_rate"] = (
        global_feat["all_sign_changes"] / (global_feat["all_n_obs"] - 1).clip(lower=1)
    ).astype(np.float32)
    global_feat["all_snr_gt8"] = global_feat["all_snr_gt8"].fillna(0).astype(np.float32)
    global_feat["all_snr_sum"] = global_feat["all_snr_sum"].fillna(0).astype(np.float32)
    global_feat["all_snr_energy"] = global_feat["all_snr_energy"].fillna(0).astype(np.float32)
    global_feat["all_snr_energy_norm"] = (
        global_feat["all_snr_energy"] / global_feat["all_n_obs"].clip(lower=1)
    ).astype(np.float32)
    global_feat["all_flux_energy"] = global_feat["all_flux_energy"].fillna(0).astype(np.float32)
    global_feat["all_flux_energy_norm"] = (
        global_feat["all_flux_energy"] / global_feat["all_n_obs"].clip(lower=1)
    ).astype(np.float32)
    global_feat["all_snr_std"] = global_feat["all_snr_std"].fillna(0).astype(np.float32)
    global_feat["all_err_std"] = global_feat["all_err_std"].fillna(0).astype(np.float32)
    global_feat["all_snr_cv"] = (
        global_feat["all_snr_std"] / global_feat["all_snr_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_err_cv"] = (
        global_feat["all_err_std"] / global_feat["all_err_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    global_feat["all_flux_wmean"] = global_feat["all_flux_wmean"].fillna(0).astype(np.float32)
    global_feat["all_flux_wstd"] = global_feat["all_flux_wstd"].fillna(0).astype(np.float32)
    global_feat["all_flux_wcv"] = (
        global_feat["all_flux_wstd"] / (global_feat["all_flux_wmean"].abs() + 1e-3)
    ).astype(np.float32)

    det_all = df_all[(df_all["snr"] >= 3) & (df_all["flux"] > 0)]
    if len(det_all) > 0:
        gd = det_all.groupby("object_id", sort=False)
        detg = gd.agg(
            det_all_n=("mjd", "count"),
            det_all_t_min=("mjd", "min"),
            det_all_t_max=("mjd", "max"),
            det_all_snr_max=("snr", "max"),
            det_all_flux_max=("flux", "max"),
        )
        detg["det_all_span"] = (detg["det_all_t_max"] - detg["det_all_t_min"]).astype(np.float32)
        detg = detg.drop(columns=["det_all_t_min", "det_all_t_max"]).reset_index()
    else:
        detg = pd.DataFrame(columns=["object_id", "det_all_n", "det_all_snr_max", "det_all_flux_max", "det_all_span"])

    global_feat = global_feat.merge(detg, on="object_id", how="left")
    global_feat["det_all_n"] = global_feat["det_all_n"].fillna(0).astype(np.float32)
    global_feat["det_all_span"] = global_feat["det_all_span"].fillna(0).astype(np.float32)
    global_feat["det_all_frac"] = (
        global_feat["det_all_n"] / global_feat["all_n_obs"].clip(lower=1)
    ).astype(np.float32)
    global_feat["det_all_span_ratio"] = (
        global_feat["det_all_span"] / global_feat["all_t_span"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

    out = global_feat.merge(wide, on="object_id", how="left")
    return out


def load_features(kind: str) -> pd.DataFrame:
    feats = []
    for sp in SPLIT_DIRS:
        path = os.path.join(sp, f"{kind}_full_lightcurves.csv")
        lc = pd.read_csv(path, dtype=LC_DTYPES)
        feats.append(extract_features_from_lc(lc))
        del lc
        gc.collect()
    return pd.concat(feats, ignore_index=True)


print("Building train features...")
train_feat = load_features("train")
print("Building test features...")
test_feat = load_features("test")

train_df = train_log.merge(train_feat, on="object_id", how="left")
test_df = test_log.merge(test_feat, on="object_id", how="left")
del train_feat, test_feat
gc.collect()


Building train features...
Building test features...


0

In [28]:

# Physics + rest-frame features


def luminosity_distance_mpc(z: np.ndarray, H0: float = 70.0, Om0: float = 0.3, n_grid: int = 7000) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    z = np.nan_to_num(z, nan=0.0)
    z = np.clip(z, 0.0, None)
    zmax = float(z.max(initial=0.0))
    if zmax <= 0:
        return np.zeros_like(z, dtype=np.float64)

    grid = np.linspace(0.0, zmax, int(n_grid))
    Ez = np.sqrt(Om0 * (1.0 + grid) ** 3 + (1.0 - Om0))
    invE = 1.0 / Ez
    dz = np.diff(grid)
    integral = np.concatenate([[0.0], np.cumsum((invE[:-1] + invE[1:]) * 0.5 * dz)])
    c = 299792.458  # km/s
    Dc = (c / H0) * integral
    Dc_z = np.interp(z, grid, Dc)
    return (1.0 + z) * Dc_z


def add_physics_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    z = df["Z"].astype(float).fillna(0.0).clip(lower=0.0).values
    inv1pz = 1.0 / (1.0 + z)
    df["inv1pz"] = inv1pz.astype(np.float32)

    dl = luminosity_distance_mpc(z)
    df["DL_Mpc"] = dl.astype(np.float32)
    df["logDL"] = np.log1p(df["DL_Mpc"]).astype(np.float32)
    df["distmod"] = (5.0 * np.log10(np.clip(df["DL_Mpc"].astype(np.float64), 1e-6, None)) + 25.0).astype(np.float32)

    time_keys = [
        "t_span",
        "peak_rel",
        "max_gap",
        "med_gap",
        "det_span",
        "width_",
        "dt_peak_",
        "dt_",
        "com",
        "all_t_span",
        "all_max_gap",
    ]
    time_cols = [c for c in df.columns if any(k in c for k in time_keys)]
    for c in time_cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c + "_rest"] = (df[c].astype(np.float32) * df["inv1pz"]).astype(np.float32)

    for c in df.columns:
        if c.endswith("_rise_slope") or c.endswith("_decay_slope") or c.endswith("_trend_slope"):
            if pd.api.types.is_numeric_dtype(df[c]):
                df[c + "_rest"] = (
                    df[c].astype(np.float32) / df["inv1pz"].replace(0, np.nan)
                ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

    dl2 = df["DL_Mpc"].astype(np.float32) ** 2
    for f in FILTERS:
        pf = f"{f}_peak_flux"
        if pf in df.columns and pd.api.types.is_numeric_dtype(df[pf]):
            L = df[pf].astype(np.float32) * dl2
            df[f"{f}_peak_L"] = L.astype(np.float32)
            df[f"{f}_peak_L_log"] = np.log1p(np.abs(L)).astype(np.float32)
            flux = np.clip(df[pf].astype(np.float64).values, 1e-3, None)
            m = 23.9 - 2.5 * np.log10(flux)
            df[f"{f}_Mpeak"] = (m - df["distmod"].astype(np.float64)).astype(np.float32)

    if "all_flux_max" in df.columns and pd.api.types.is_numeric_dtype(df["all_flux_max"]):
        flux = np.clip(df["all_flux_max"].astype(np.float64).values, 1e-3, None)
        m = 23.9 - 2.5 * np.log10(flux)
        df["Mall_max"] = (m - df["distmod"].astype(np.float64).values).astype(np.float32)

    if "Z_err" in df.columns:
        ze = df["Z_err"].astype(float).values
        zsnr = np.divide(df["Z"].astype(float).values, ze, out=np.zeros_like(ze), where=(ze > 0))
        df["Z_snr"] = np.nan_to_num(zsnr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    eps = 1e-6
    for f in FILTERS:
        q25, q75, med = f"{f}_flux_q25", f"{f}_flux_q75", f"{f}_flux_median"
        if all(c in df.columns for c in [q25, q75, med]):
            denom = (df[q75] - df[q25]).astype(np.float32).replace(0, np.nan)
            df[f"{f}_bowley_skew"] = (
                (df[q75] + df[q25] - 2 * df[med]).astype(np.float32) / denom
            ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

        if all(c in df.columns for c in [f"{f}_flux_q10", f"{f}_flux_q90", f"{f}_flux_iqr"]):
            denom = df[f"{f}_flux_iqr"].astype(np.float32).replace(0, np.nan)
            df[f"{f}_tail_9010"] = (
                (df[f"{f}_flux_q90"] - df[f"{f}_flux_q10"]).astype(np.float32) / denom
            ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

        if all(c in df.columns for c in [f"{f}_peak_flux", med, f"{f}_flux_iqr"]):
            denom = df[f"{f}_flux_iqr"].astype(np.float32).replace(0, np.nan)
            df[f"{f}_peakiness"] = (
                (df[f"{f}_peak_flux"] - df[med]).astype(np.float32) / denom
            ).replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

    blue_cols = [c for c in ["u_peak_flux", "g_peak_flux"] if c in df.columns]
    red_cols = [c for c in ["i_peak_flux", "z_peak_flux", "y_peak_flux"] if c in df.columns]
    if blue_cols and red_cols:
        blue = df[blue_cols].sum(axis=1).astype(np.float32)
        red = df[red_cols].sum(axis=1).astype(np.float32)
        df["blue_red_asinh"] = (np.arcsinh(blue.astype(np.float64)) - np.arcsinh(red.astype(np.float64))).astype(
            np.float32
        )

    return df


train_df = add_physics_features(train_df)
test_df = add_physics_features(test_df)

for df in [train_df, test_df]:
    df["Z"] = df["Z"].astype(np.float32)
    df["Z_err"] = df["Z_err"].astype(np.float32)
    df["EBV"] = df["EBV"].astype(np.float32)
    df["log1pZ"] = np.log1p(df["Z"].clip(lower=0)).astype(np.float32)
    df["Z2"] = (df["Z"] * df["Z"]).astype(np.float32)
    df["EBV2"] = (df["EBV"] * df["EBV"]).astype(np.float32)
    df["EBV_Z"] = (df["EBV"] * df["Z"]).astype(np.float32)
    df["EBV_log1pZ"] = (df["EBV"] * df["log1pZ"]).astype(np.float32)
    for cat_col in ["first_filter", "peak_filter"]:
        if cat_col in df.columns:
            df[cat_col] = df[cat_col].fillna("__NONE__").astype("category")


In [29]:

# Model

DO_TUNING = False
TUNING_TRIALS = 20
TUNING_FOLDS = 3
TUNING_EARLY_STOPPING = 200
TUNING_N_ESTIMATORS = 2500
EARLY_STOPPING_ROUNDS = 300


def best_threshold_pr(y_true: np.ndarray, prob: np.ndarray) -> Tuple[float, float]:
    prec, rec, thr = precision_recall_curve(y_true, prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    idx = int(np.argmax(f1[:-1]))
    return float(thr[idx]), float(f1[idx])


def prep_X(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["target"].astype(int).values
    drop_cols_train = [c for c in ["target", "SpecType", "English Translation", "object_id"] if c in train_df.columns]
    drop_cols_test = [c for c in ["SpecType", "English Translation", "object_id"] if c in test_df.columns]

    X = train_df.drop(columns=drop_cols_train).copy()
    X_test = test_df.drop(columns=drop_cols_test).copy()

    if "split" in X.columns:
        X["split"] = X["split"].astype("category")
        X_test["split"] = X_test["split"].astype("category")

    num_cols = X.select_dtypes(include=[np.number]).columns
    med = X[num_cols].median()
    X[num_cols] = X[num_cols].fillna(med)
    X_test[num_cols] = X_test[num_cols].fillna(med)

    cat_cols = X.select_dtypes(include=["category"]).columns
    for c in cat_cols:
        X[c] = X[c].cat.add_categories(["__MISSING__"]).fillna("__MISSING__")
        if c in X_test.columns:
            X_test[c] = X_test[c].cat.add_categories(["__MISSING__"]).fillna("__MISSING__")

    nunique = X.nunique(dropna=False)
    const_cols = nunique[nunique <= 1].index.tolist()
    if const_cols:
        X = X.drop(columns=const_cols)
        X_test = X_test.drop(columns=const_cols)

    return X, y, X_test


def sample_param_grid(param_grid: dict, n_samples: int, seed: int) -> List[dict]:
    keys = list(param_grid)
    rng = np.random.default_rng(seed)
    total = int(np.prod([len(param_grid[k]) for k in keys]))
    if n_samples >= total:
        return list(ParameterGrid(param_grid))

    seen = set()
    samples: List[dict] = []
    while len(samples) < n_samples and len(seen) < total:
        cand = {k: rng.choice(v) for k, v in param_grid.items()}
        key = tuple((k, cand[k]) for k in keys)
        if key in seen:
            continue
        seen.add(key)
        samples.append(cand)
    return samples


def cv_f1_score(
    X: pd.DataFrame,
    y: np.ndarray,
    params: dict,
    n_splits: int = 3,
    seed: int = 42,
    early_stopping_rounds: int = 200,
) -> float:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cat_features = list(X.select_dtypes(include=["category"]).columns)
    f1s: List[float] = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = lgb.LGBMClassifier(**params, random_state=seed + 1000 * fold)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            categorical_feature=cat_features,
            callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
        )
        p_va = model.predict_proba(X_va)[:, 1]
        _, f1b = best_threshold_pr(y_va, p_va)
        f1s.append(f1b)

    return float(np.mean(f1s))


def tune_lgb_params(
    X: pd.DataFrame,
    y: np.ndarray,
    base_params: dict,
    n_trials: int = 20,
    n_splits: int = 3,
    seed: int = 42,
    early_stopping_rounds: int = 200,
    n_estimators: int = 2500,
) -> Tuple[pd.DataFrame, dict]:
    param_grid = {
        "num_leaves": [63, 95, 127, 191],
        "min_child_samples": [10, 20, 40, 80],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.6, 0.75, 0.9],
        "feature_fraction_bynode": [0.6, 0.75, 0.85],
        "reg_alpha": [0.0, 0.2, 0.6, 1.0],
        "reg_lambda": [2.0, 6.0, 10.0],
        "min_split_gain": [0.0, 0.01, 0.05],
        "max_depth": [-1, 8, 12],
    }

    candidates = sample_param_grid(param_grid, n_trials, seed)
    results = []
    for i, cand in enumerate(candidates, 1):
        params = base_params.copy()
        params.update(cand)
        params["n_estimators"] = n_estimators
        score = cv_f1_score(
            X,
            y,
            params,
            n_splits=n_splits,
            seed=seed,
            early_stopping_rounds=early_stopping_rounds,
        )
        row = cand.copy()
        row["mean_f1"] = score
        results.append(row)
        print(f"Trial {i}/{len(candidates)}: F1={score:.5f} | {cand}")

    results_df = pd.DataFrame(results).sort_values("mean_f1", ascending=False)
    best_params = {}
    if not results_df.empty:
        best_params = results_df.iloc[0].drop(labels=["mean_f1"]).to_dict()
    return results_df, best_params


def train_cv_lgb(
    X: pd.DataFrame,
    y: np.ndarray,
    X_test: pd.DataFrame,
    params: dict,
    n_splits: int = 5,
    seed: int = 42,
    early_stopping_rounds: int = 300,
) -> Tuple[np.ndarray, np.ndarray, float, float]:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X), dtype=np.float32)
    pred_test = np.zeros(len(X_test), dtype=np.float32)
    fold_ts: List[float] = []
    fold_f1s: List[float] = []
    cat_features = list(X.select_dtypes(include=["category"]).columns)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = lgb.LGBMClassifier(**params, random_state=seed + 1000 * fold)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            categorical_feature=cat_features,
            callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
        )

        p_va = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = p_va
        pred_test += model.predict_proba(X_test)[:, 1] / n_splits

        t, f1b = best_threshold_pr(y_va, p_va)
        fold_ts.append(t)
        fold_f1s.append(f1b)
        print(f"Fold {fold}: F1={f1b:.5f} | thr={t:.4f}")

    median_thr = float(np.median(fold_ts))
    global_thr, global_f1 = best_threshold_pr(y, oof)
    best_t = float(global_thr)
    if best_t <= 0 or best_t >= 1:
        best_t = median_thr
    oof_f1 = float(global_f1)
    print(f"OOF F1={oof_f1:.5f} | thr={best_t:.4f} (median_thr={median_thr:.4f})")
    return oof, pred_test, best_t, oof_f1


X, y, X_test = prep_X(train_df, test_df)
pos = int(y.sum())
neg = int(len(y) - pos)
scale_pos_weight = neg / max(pos, 1)
print(f"pos={pos} neg={neg} scale_pos_weight={scale_pos_weight:.2f}")

params = dict(
    n_estimators=6500,
    learning_rate=0.02,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.75,
    feature_fraction_bynode=0.8,
    reg_lambda=6.0,
    reg_alpha=0.4,
    min_child_samples=20,
    min_split_gain=0.01,
    objective="binary",
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    extra_trees=True,
    force_col_wise=True,
)

if DO_TUNING:
    tune_df, best_params = tune_lgb_params(
        X,
        y,
        params,
        n_trials=TUNING_TRIALS,
        n_splits=TUNING_FOLDS,
        seed=RANDOM_STATE,
        early_stopping_rounds=TUNING_EARLY_STOPPING,
        n_estimators=TUNING_N_ESTIMATORS,
    )
    if not tune_df.empty:
        print(tune_df.head(5))
        params.update(best_params)

oof, test_prob, best_thr, oof_f1 = train_cv_lgb(
    X,
    y,
    X_test,
    params,
    n_splits=N_FOLDS,
    seed=RANDOM_STATE,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

final_f1 = float(f1_score(y, (oof >= best_thr).astype(int)))
print(f"[FINAL] OOF F1={final_f1:.5f} | thr={best_thr:.4f}")

sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
pred_col = "prediction" if "prediction" in sub.columns else sub.columns[-1]
sub[pred_col] = (test_prob >= best_thr).astype(int)
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(sub.head())


pos=148 neg=2895 scale_pos_weight=19.56
[LightGBM] [Info] Number of positive: 118, number of negative: 2316
[LightGBM] [Info] Total Bins 244440
[LightGBM] [Info] Number of data points in the train set: 2434, number of used features: 1001
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.048480 -> initscore=-2.976912
[LightGBM] [Info] Start training from score -2.976912
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt